# Projet IC1 (3/3) — Interprétation et carte

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/inphb/ic1/notebooks/IC1_Projet_S5_interpretation.ipynb)
> Données : dossier `IC1_TP_donnees`. Bibliothèques : pandas, numpy, matplotlib, scikit-learn.

In [ ]:
# Configuration : Colab ou Jupyter local. Exécutez cette cellule en premier.
import os, shutil, urllib.request
DONNEES = ['composites_2m.csv']
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/inphb/ic1/data/'
for f in DONNEES:
    if os.path.exists(f):
        continue
    if os.path.exists(os.path.join('..', 'data', f)):
        shutil.copy(os.path.join('..', 'data', f), f)   # kit hors ligne / dépôt cloné
    else:
        urllib.request.urlretrieve(URL + f, f)            # Colab : téléchargement
    print('OK', f)


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold, cross_val_score, cross_val_predict, learning_curve, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
comp = pd.read_csv('composites_2m.csv'); comp['mineralise'] = (comp['au_gpt'] > 0.5).astype(int)
num, cat = ['sulfures_pct', 'alteration', 'de_m'], ['lithologie']; X, y, g = comp[num + cat], comp['mineralise'], comp['sondage']
prep = ColumnTransformer([('num', StandardScaler(), num), ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])
pipe = Pipeline([('prep', prep), ('m', RandomForestClassifier(400, min_samples_leaf=5, random_state=0))])

## 1. Importance par permutation (sur des sondages tenus à l’écart)

In [ ]:
tr, te = next(GroupKFold(5).split(X, y, g))
pipe.fit(X.iloc[tr], y.iloc[tr])
r = permutation_importance(pipe, X.iloc[te], y.iloc[te], scoring='roc_auc', n_repeats=10, random_state=0)
print(pd.Series(r.importances_mean, index=X.columns).sort_values(ascending=False).round(3))

## 2. Carte : probabilité prédite (sondage jamais vu) vs teneur mesurée

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
gb = Pipeline([('prep', prep), ('m', HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, random_state=0))])
p = cross_val_predict(gb, X, y, cv=GroupKFold(5), groups=g, method='predict_proba')[:, 1]
comp['p'] = p
h = comp.groupby('sondage').agg(x=('x_m', 'first'), y=('y_m', 'first'), p=('p', 'mean'), au=('au_gpt', 'mean'))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(h.x, h.y, c=h.p, cmap='YlOrRd', s=90); ax[0].set_title('P prédite')
ax[1].scatter(h.x, h.y, c=np.log10(h.au), cmap='YlOrRd', s=90); ax[1].set_title('Teneur mesurée'); plt.show()
print('corrélation :', round(np.corrcoef(h.p, np.log10(h.au))[0, 1], 2))

## 3. Rapport
Suivre la structure du module 6 : besoin, données, méthode, résultats, interprétation, **limites** (domaine de validité, sondages à ajouter), annexes (versions, graines).